# Two-Dimensional Laminar Boundary Layer Flow and Hydrodynamic Instability

This tutorial demonstrates using **`symlie`** to analyze:
1. **Prandtl's 2D Steady Laminar Boundary Layer Flow** on a flat plate and its **Blasius self-similarity Lie group reduction**.
2. **Hydrodynamic Linear Instability Analysis**: Derivation of the **Rayleigh Instability Equation** and **Orr–Sommerfeld Operator** via Fréchet linearization on jet spaces.

In [ ]:
import sympy as sp

from symlie import (
    InfinitesimalGenerator,
    frechet_derivative,
    lie_bracket,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

## Part I: 2D Laminar Boundary Layer Flow & Blasius Symmetry Reduction

### 1. The Boundary Layer Equation in Streamfunction Form

For steady, 2D incompressible laminar flow over a flat plate with zero pressure gradient, Prandtl's boundary layer equations are:
$$u u_x + v u_y = \nu u_{yy}, \quad u_x + v_y = 0$$

Introducing the streamfunction $\psi(x, y)$ such that $u = \psi_y$ and $v = -\psi_x$, the continuity equation is automatically satisfied ($u_x + v_y = \psi_{yx} - \psi_{xy} = 0$), and the momentum equation becomes the 3rd-order nonlinear PDE:
$$\Delta_{\text{BL}} = \psi_y \psi_{xy} - \psi_x \psi_{yy} - \nu \psi_{yyy} = 0$$

In [ ]:
x, y = sp.symbols("x y", positive=True)
nu = sp.symbols("nu", positive=True)
psi = sp.Function("psi")(x, y)

# 2D Boundary Layer Equation in terms of streamfunction psi(x, y)
bl_eq = (
    psi.diff(y) * psi.diff(x, y) - psi.diff(x) * psi.diff(y, 2) - nu * psi.diff(y, 3)
)

print("Boundary Layer PDE Order:", max_derivative_order(bl_eq, psi, (x, y)))
sp.Eq(bl_eq, 0)

### 2. Lie Symmetries of the Boundary Layer Equation

The 2D boundary layer equation admits the following classical Lie point symmetries:
1. **Streamwise Translation**: $X_1 = \frac{\partial}{\partial x}$
2. **Streamfunction Gauge Shift**: $X_2 = \frac{\partial}{\partial \psi}$
3. **Blasius Scaling / Self-Similarity Symmetry**:
   $$X_{\text{Blasius}} = 2x \frac{\partial}{\partial x} + y \frac{\partial}{\partial y} + \psi \frac{\partial}{\partial \psi}$$
   which corresponds to the scaling group: $(x, y, \psi) \mapsto (\lambda^2 x, \lambda y, \lambda \psi)$.

In [ ]:
# Define infinitesimal generators
X_trans = InfinitesimalGenerator(xi=(1, 0), phi=(0,))
X_gauge = InfinitesimalGenerator(xi=(0, 0), phi=(1,))
X_blasius = InfinitesimalGenerator(xi=(2 * x, y), phi=(psi,))

# Verify that each generator satisfies the prolongation invariance condition pr^(3) X(Delta) = 0
print(
    "X_1 (Translation in x) Invariant:", verify_generator(bl_eq, psi, (x, y), X_trans)
)
print(
    "X_2 (Gauge shift in psi) Invariant:", verify_generator(bl_eq, psi, (x, y), X_gauge)
)
print(
    "X_Blasius (Self-similarity) Invariant:",
    verify_generator(bl_eq, psi, (x, y), X_blasius),
)

### 3. Commutator Algebra of Boundary Layer Symmetries

We compute the Lie bracket $[X_1, X_{\text{Blasius}}]$:

In [ ]:
bracket_1_blasius = lie_bracket(X_trans, X_blasius, psi, (x, y))
print("[X_1, X_Blasius] =", bracket_1_blasius)
assert bracket_1_blasius.xi == (2, 0) and bracket_1_blasius.phi == (0,)
print("Commutator verified: [X_1, X_Blasius] = 2 X_1")

### 4. Symmetry Reduction: The Blasius Equation

The invariant surface condition for $X_{\text{Blasius}} = 2x \partial_x + y \partial_y + \psi \partial_\psi$ is:
$$\frac{dx}{2x} = \frac{dy}{y} = \frac{d\psi}{\psi}$$

Integrating yields the **similarity variable** $\eta$ and **similarity form** $\psi(x, y)$:
$$\eta = \frac{y}{\sqrt{2\nu x}}, \quad \psi(x, y) = \sqrt{2\nu x}\, f(\eta)$$

Substituting $\psi(x, y) = \sqrt{2\nu x}\, f(\eta)$ into the boundary layer equation reduces the PDE to the famous **Blasius ODE**:
$$f'''+ f\, f'' = 0$$
with boundary conditions $f(0) = f'(0) = 0$ (no-slip and impermeability at the plate) and $f'(\infty) = 1$ (matching free-stream velocity).


In [ ]:
eta = sp.symbols("eta", positive=True)
f = sp.Function("f")(eta)

# Similarity ansatz: psi(x, y) = sqrt(2 * nu * x) * f(y / sqrt(2 * nu * x))
psi_ansatz = sp.sqrt(2 * nu * x) * f.subs(eta, y / sp.sqrt(2 * nu * x))
reduced_blasius = sp.simplify(-2 * x * bl_eq.subs(psi, psi_ansatz).doit())
reduced_blasius = sp.simplify(reduced_blasius.subs(y, eta * sp.sqrt(2 * nu * x)).doit())
expected_blasius = f.diff(eta, 3) + f * f.diff(eta, 2)
assert sp.simplify(reduced_blasius - expected_blasius) == 0
print("Blasius reduction verified:")
display(sp.Eq(reduced_blasius, 0))

# Compute velocity components
u_sim = psi_ansatz.diff(y)
v_sim = -psi_ansatz.diff(x)

print("u(x, y) =")
display(u_sim)
print("v(x, y) =")
display(v_sim)

## Part II: Hydrodynamic Instability & the Rayleigh / Orr–Sommerfeld Equation

To analyze whether a laminar boundary layer velocity profile $U(y)$ becomes unstable to small disturbances, we linearize the 2D Navier–Stokes vorticity equation around the base shear flow.

### 1. 2D Navier–Stokes Vorticity Transport Equation
The 2D vorticity equation for $\omega = -\nabla^2 \psi = -(\psi_{xx} + \psi_{yy})$ is:
$$\frac{\partial \omega}{\partial t} + u \frac{\partial \omega}{\partial x} + v \frac{\partial \omega}{\partial y} = \nu \nabla^2 \omega$$

In terms of streamfunction $\psi(x, y, t)$:
$$-\nabla^2 \psi_t - \psi_y \nabla^2 \psi_x + \psi_x \nabla^2 \psi_y + \nu \nabla^4 \psi = 0$$

In [ ]:
t = sp.symbols("t")
psi_tot = sp.Function("psi")(x, y, t)
lap_psi = psi_tot.diff(x, 2) + psi_tot.diff(y, 2)

# Vorticity equation
vort_eq = (
    -lap_psi.diff(t)
    - psi_tot.diff(y) * lap_psi.diff(x)
    + psi_tot.diff(x) * lap_psi.diff(y)
    + nu * (lap_psi.diff(x, 2) + lap_psi.diff(y, 2))
)

print("Vorticity Equation Order:", max_derivative_order(vort_eq, psi_tot, (x, y, t)))
sp.Eq(vort_eq, 0)

### 2. Fréchet Linearization around Base Shear Flow $U(y)$

Let $\psi(x, y, t) = \psi_0(y) + \epsilon \tilde{\psi}(x, y, t)$, where the base parallel flow velocity is $U(y) = \psi_0'(y)$.

The linear disturbance equation is given by the **Fréchet derivative** acting on the disturbance streamfunction $\tilde{\psi}(x, y, t)$:

In [ ]:
phi_dist = sp.Function("phi")(x, y, t)

# Compute the Fréchet derivative of the vorticity equation
D_vort = frechet_derivative(vort_eq, psi_tot, (x, y, t), phi_dist)
linearized_pde = D_vort[0, 0]

print("Linearized Disturbance Operator acting on phi(x, y, t):")
display(linearized_pde)

### 3. Normal Modes: The Rayleigh and Orr–Sommerfeld Equations

Substituting a 2D Fourier normal mode perturbation:
$$\tilde{\psi}(x, y, t) = \hat{\phi}(y) e^{i(\alpha x - \omega t)} = \hat{\phi}(y) e^{i \alpha (x - c t)}$$
where $\alpha$ is the real streamwise wavenumber and $c = \omega / \alpha = c_r + i c_i$ is the complex phase speed (with $c_i > 0$ representing modal exponential growth / instability).

In the **inviscid limit** ($\nu \to 0$, high Reynolds number), the linearized operator reduces to the **Rayleigh Stability Equation**:
$$(U(y) - c) (\hat{\phi}''(y) - \alpha^2 \hat{\phi}(y)) - U''(y) \hat{\phi}(y) = 0$$

In the **viscous case**, it yields the 4th-order **Orr–Sommerfeld Equation**:
$$(U - c)(\hat{\phi}'' - \alpha^2 \hat{\phi}) - U'' \hat{\phi} = -\frac{i}{\alpha \text{Re}} (\hat{\phi}'''' - 2\alpha^2 \hat{\phi}'' + \alpha^4 \hat{\phi})$$

In [ ]:
alpha = sp.symbols("alpha", real=True, nonzero=True)
c = sp.symbols("c", complex=True)
phi_y = sp.Function("phi")(y)
U = sp.Function("U")(y)

base_streamfunction = sp.Integral(U, y)
linearized_base = sp.simplify(linearized_pde.subs(psi_tot, base_streamfunction).doit())
normal_mode = phi_y * sp.exp(sp.I * alpha * (x - c * t))
normal_residual = sp.simplify(
    sp.expand(linearized_base.subs(phi_dist, normal_mode).doit())
    / sp.exp(sp.I * alpha * (x - c * t))
)
biharmonic_mode = phi_y.diff(y, 4) - 2 * alpha**2 * phi_y.diff(y, 2) + alpha**4 * phi_y
# Rayleigh Inviscid Instability Operator
rayleigh_eq = (U - c) * (phi_y.diff(y, 2) - alpha**2 * phi_y) - U.diff(y, 2) * phi_y
orr_sommerfeld_eq = rayleigh_eq + sp.I * nu / alpha * biharmonic_mode
assert sp.simplify(normal_residual + sp.I * alpha * orr_sommerfeld_eq) == 0

print("Rayleigh Stability Equation:")
display(sp.Eq(rayleigh_eq, 0))
print("Orr-Sommerfeld Equation (with nu = 1/Re after nondimensionalization):")
display(sp.Eq(orr_sommerfeld_eq, 0))

### 4. Rayleigh's Inflection Point Theorem

Multiply the Rayleigh equation by $\hat{\phi}^* / (U - c)$ and integrate across the boundary layer $y \in [0, \infty)$ with $\hat{\phi}(0) = \hat{\phi}(\infty) = 0$:
$$\int_0^\infty \left( |\hat{\phi}'|^2 + \alpha^2 |\hat{\phi}|^2 \right) dy + \int_0^\infty \frac{U''(y)}{U(y) - c} |\hat{\phi}|^2 dy = 0$$

Taking the imaginary part (with $c = c_r + i c_i$):
$$c_i \int_0^\infty \frac{U''(y)}{|U(y) - c|^2} |\hat{\phi}(y)|^2 dy = 0$$

**Theorem (Lord Rayleigh, 1880)**: For an unstable mode ($c_i > 0$) to exist in an inviscid shear flow, the velocity profile $U(y)$ must contain at least one **inflection point** $U''(y_s) = 0$ inside the boundary layer.

- **Fjørtoft's Extension (1950)**: In addition to $U''(y_s) = 0$, instability requires $U''(y) (U(y) - U(y_s)) < 0$ (the inflection point must be a vorticity maximum).

In [ ]:
print("Summary of Results:")
print(
    "1. Prandtl's 2D boundary layer equation admits the Blasius scaling symmetry X_Blasius."
)
print(
    "2. Symmetry reduction yields the classical Blasius self-similar boundary layer profile f''' + f f'' = 0."
)
print(
    "3. Base-flow and normal-mode substitution derives the Orr-Sommerfeld operator and its Rayleigh limit."
)
print(
    "4. Rayleigh's inflection-point criterion follows analytically from the displayed imaginary-part integral identity."
)